In [1]:
import os
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
sys.path.append("..")
from src.create_Race_minimal_pair import create_race_minimal_pairs_year

create_race_minimal_pairs_year("../data/offense.csv")

Saved minimal pairs to ../data/race_minimal_pairs_new_year.tsv
Total sentences generated: 25110


In [3]:
import matplotlib.pyplot as plt
import pandas as pd
import os
import numpy as np
from matplotlib.patches import Patch

df = pd.read_csv("../results/race_results_new_year.tsv", sep="\t")
df1 = df[(df['model'] == "distilbert-base-uncased") & (df['severity'] == 'M5')]

# Aggregate
bar_df = df1.groupby(["race", "punishment"])["acc"].mean().reset_index()

bar_data = bar_df.pivot(index="punishment", columns="race", values="acc")

try:
    races_to_plot = sorted(bar_data.columns) # e.g., ['Black', 'Hispanic', 'White']
    bar_data = bar_data[races_to_plot]
    
    labels_bottom = ["Black", "White", "White"] 
    labels_top = ["Hispanic", "Black", "Hispanic"]

except KeyError:
    print("Warning: Could not find all required columns ('Black', 'Hispanic', 'White').")
    races_to_plot = [col for col in bar_data.columns if col in label_map]
    bar_data = bar_data[races_to_plot]

x_positions_for_punishments = np.arange(len(bar_data.index)) # [0, 1, 2]
num_races = len(bar_data.columns)
bar_width = 0.25

label_map = {
    "Black": "purple",
    "White": "red",
    "Hispanic": "green"
}


fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(8, 5), sharey=True) 
axes = axes.flatten() 

# --- Loop to draw the same plot on each subplot ---

temp = ['M1','M9','D1','D4']

crimes = ["Murder", "Fraud" , "Distribution of Drug" , "Possesion of Drug"]
level = ["Serious","Light","Serious", "Light"]
for plot_idx, ax in enumerate(axes): 
    df1 = df[(df['model'] == "distilbert-base-uncased") & (df['severity'] == temp[plot_idx])]
    bar_df = df1.groupby(["race", "punishment"])["acc"].mean().reset_index()
    
    bar_data = bar_df.pivot(index="punishment", columns="race", values="acc")
    
    # --- Loop over the RACES to plot bar groups ---
    for i, race in enumerate(bar_data.columns):
        acc_values = bar_data[race]
        comp_values = 1 - acc_values

        # Get the i-th label and its color
        label_bottom = labels_bottom[i]
        label_top = labels_top[i]
        color_bottom = label_map[label_bottom]
        color_top = label_map[label_top]
        
        bar_x_positions = x_positions_for_punishments - (bar_width * num_races / 2) + (i + 0.5) * bar_width

        bars = ax.bar(bar_x_positions, acc_values, width=bar_width, 
                       color=color_bottom, edgecolor="black", 
                       label=race)

        con_bars = ax.bar(bar_x_positions, comp_values, width=bar_width,
                           bottom=acc_values, alpha=0.4, 
                           color=color_top, edgecolor="black")

        labels_bottom_list = [label_bottom] * len(acc_values)
        ax.bar_label(bars, labels=labels_bottom_list, label_type='center', 
                      color='white', fontsize=4, fontweight='bold')
        
        labels_top_list = [label_top] * len(acc_values)
        labels_top_visible = [l if c > 0.1 else "" for l, c in zip(labels_top_list, comp_values)]
        ax.bar_label(con_bars, labels=labels_top_visible, label_type='center', 
                      color='white', fontsize=4, fontweight='bold')

    # --- Customize each subplot ---
    ax.set_xticks(x_positions_for_punishments) 
    ax.set_xticklabels(bar_data.index)
    ax.set_xlabel("Punishment (Years)")
    ax.set_ylabel("Incarceration Proportion")
    ax.set_title(f"{level[plot_idx]} Crime eg( {crimes[plot_idx]})") 
    ax.set_ylim(0, 1.1)
    ax.axhline(0.5, linestyle="--", color="gray", linewidth=1)
    print(bar_data)



plt.suptitle("Racial Bias on Distilbert-base-uncased masked Model", fontsize=16, y=1.0) # Lowered y
# --- *** FIX 2: Adjust tight_layout rect *** ---
plt.tight_layout(rect=[0, 0.03, 0.97, 0.96]) # Adjusted top and right padding

os.makedirs("../assets", exist_ok=True)
plt.savefig("../assets/Distilbert_Uncased1.png", dpi=300, bbox_inches="tight")
plt.close()

print("2x2 subplot grid (Shrunk) saved!")

race        Black vs Hispanic  White vs Black  White vs Hispanic
punishment                                                      
2                        0.44            0.40               0.40
7                        0.56            0.12               0.44
15                       0.52            0.20               0.44
race        Black vs Hispanic  White vs Black  White vs Hispanic
punishment                                                      
2                       0.390            0.40              0.385
7                       0.545            0.18              0.490
15                      0.555            0.20              0.485
race        Black vs Hispanic  White vs Black  White vs Hispanic
punishment                                                      
2                    0.200000        0.333333                0.2
7                    0.266667        0.266667                0.2
15                   0.266667        0.333333                0.2
race        Black vs Hisp